In [1]:
import boto3

def create_kinesis_stream(stream_name, shard_count):
    try:
        # Initialize the kinesis client
        kinesis_client = boto3.client('kinesis')

        # Create the kinesis stream
        response = kinesis_client.create_stream(
            StreamName=stream_name,
            ShardCount=shard_count
        )

        # Check for successfull response
        if response['ResponseMetadata']['HTTPStatusCode'] == 200:
            print(f"Kinesis stream {stream_name} is created with {shard_count} shard(s)")
        else:
            print("Failed to create kinesis stream")

    except Exception as e:
        print(F"Error: {str(e)}")

def delete_kinesis_stream(stream_name):
    try:
        # Initialize the kinesis client
        kinesis_client = boto3.client('kinesis')

        # Delete the kinesis stream
        response = kinesis_client.delete_stream(StreamName=stream_name)

        # Check for successfull response
        if response['ResponseMetadata']['HTTPStatusCode'] == 200:
            print(f"Kinesis stream {stream_name} is deleted.")
        else:
            print("Failed to delete kinesis stream")
            
    except Exception as e:
        print(F"Error: {str(e)}")

In [2]:
# create_kinesis_stream('input-stream', 1)
delete_kinesis_stream('input-stream')

Kinesis stream input-stream is deleted.


### Publish faker data

In [12]:
import datetime
import json
import random
import sys

import boto3
from botocore.exceptions import BotoCoreError, ClientError

# Configuration
STREAM_NAME = "input-stream"
AWS_REGION = "us-east-1"
NUM_SAMPLES = 10


def get_data():
    """Generate a dummy stock price record."""
    return {
        "event_time": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "ticker": random.choice(["AAPL", "AMZN", "MSFT", "INTC", "TBV"]),
        "price": round(random.uniform(0, 100), 2),
    }


def generate(stream_name, kinesis_client, num_samples):
    """Publish dummy records to an AWS Kinesis stream."""
    for i in range(num_samples):
        data = get_data()

        try:
            response = kinesis_client.put_record(
                StreamName=stream_name,
                Data=json.dumps(data),
                PartitionKey=data["ticker"],
            )

            print(f"[{i + 1}/{num_samples}] Published: {json.dumps(data)}")
            print(f"    Shard ID: {response['ShardId']}")
            print(f"    Sequence Number: {response['SequenceNumber']}")

        except (ClientError, BotoCoreError) as error:
            print(f"Error publishing record {i + 1}: {error}", file=sys.stderr)
            raise


def main():
    """Validate the stream and publish records."""
    kinesis_client = boto3.client("kinesis", region_name=AWS_REGION)

    try:
        response = kinesis_client.describe_stream_summary(
            StreamName=STREAM_NAME
        )
        stream_status = response["StreamDescriptionSummary"]["StreamStatus"]

        if stream_status != "ACTIVE":
            print(
                f"Stream '{STREAM_NAME}' is not active. "
                f"Current status: {stream_status}"
            )
            return

        print(f"Stream '{STREAM_NAME}' is ACTIVE.")
        print(f"Publishing {NUM_SAMPLES} records...\n")

        generate(STREAM_NAME, kinesis_client, NUM_SAMPLES)
        print("\nAll records published successfully.")

    except (ClientError, BotoCoreError) as error:
        print(f"AWS error: {error}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Stream 'input-stream' is ACTIVE.
Publishing 10 records...

[1/10] Published: {"event_time": "2026-11-02T20:17:00.218484+00:00", "ticker": "AMZN", "price": 78.59}
    Shard ID: shardId-000000000000
    Sequence Number: 49679853119687095886109373919859920712314720416144818178
[2/10] Published: {"event_time": "2026-11-02T20:17:00.556522+00:00", "ticker": "AAPL", "price": 5.74}
    Shard ID: shardId-000000000000
    Sequence Number: 49679853119687095886109373919978395442636954075266023426
[3/10] Published: {"event_time": "2026-11-02T20:17:00.934431+00:00", "ticker": "AMZN", "price": 8.31}
    Shard ID: shardId-000000000000
    Sequence Number: 49679853119687095886109373920107750505335719465679060994
[4/10] Published: {"event_time": "2026-11-02T20:17:01.268591+00:00", "ticker": "MSFT", "price": 42.43}
    Shard ID: shardId-000000000000
    Sequence Number: 49679853119687095886109373920203255645085275170480848898
[5/10] Published: {"event_time": "2026-11-02T20:17:01.651131+00:00", "ticker": 

### Python flink code
Download the connector jar file for flink

https://mvnrepository.com/artifact/org.apache.flink/flink-sql-connector-kinesis/1.16.1

In [26]:
import os

os.listdir('.')

['01-TableAPI-Intro.ipynb',
 '__init__.py',
 'flink-sql-connector-kinesis_2.12-1.13.1.jar',
 '02-pyflink-with-kinesis.ipynb',
 '.ipynb_checkpoints',
 'flink-sql-connector-kinesis-1.16.1.jar']

In [27]:
from pyflink.table import TableEnvironment, EnvironmentSettings
import os


# 1. Create table environment
env_settings = EnvironmentSettings.in_streaming_mode()
table_env = TableEnvironment.create(environment_settings=env_settings)

# 2. Define configuration properties for flink job
props = [
    {
        'PropertyGroupId': 'kinesis.analytics.flink.run.options',
        'PropertyMap': {
            'python': 'GettingStarted/getting-started.py',
            'jarfile': 'flink-sql-connector-kinesis_2.12-1.13.1.jar'
        }
    },
    {
        'PropertyGroupId': 'consumer.config.0',
        'PropertyMap': {
            'input.stream.name': 'input-stream',
            'flink.stream.initpos': 'TRIM_HORIZON',
            'aws.region': 'us-east-1'
        }
    }
]

# Function to extract property map from the configuration properties
def property_map(props, property_group_id):
    for prop in props:
        if prop["PropertyGroupId"] == property_group_id:
            return prop["PropertyMap"]


# Function to create a source table definition for Kinesis stream
def create_source_table(table_name, stream_name, region, stream_initpos):
    return f"""
        CREATE TABLE {table_name} (
            ticker VARCHAR(6),
            price DOUBLE,
            event_time TIMESTAMP(3),
            WATERMARK FOR event_time AS event_time - INTERVAL '5' SECOND
        )
        PARTITIONED BY (ticker)
        WITH (
            'connector' = 'kinesis',
            'stream' = '{stream_name}',
            'aws.region' = '{region}',
            'scan.stream.initpos' = '{stream_initpos}',
            'format' = 'json',
            'json.timestamp-format.standard' = 'ISO-8601'
        )
    """

In [28]:
# Main function
def main():
    # Define keys for accessing properties
    input_property_group_key = "consumer.config.0"
    input_stream_key = "input.stream.name"
    input_region_key = "aws.region"
    input_starting_position_key = "flink.stream.initpos"

    # Table name for the source data
    input_table_name = "input_table"

    # Get application properties from the props list
    input_property_map = property_map(props, input_property_group_key)
    input_stream = input_property_map[input_stream_key]
    input_region = input_property_map[input_region_key]
    stream_initpos = input_property_map[input_starting_position_key]

    # Check if running in a local environment and set pipeline.jars configuration
    CURRENT_DIR = os.getcwd()  # Get the current working directory

    table_env.get_config().get_configuration().set_string(
        "pipeline.jars",
        "file:///" + CURRENT_DIR + "/flink-sql-connector-kinesis_2.12-1.13.1.jar",
    )

    # Create the table
    table_env.execute_sql(
        create_source_table(
            input_table_name, input_stream, input_region, stream_initpos
        )
    )

    print('Successfully create table')

    # 3. Read data from the source table and print it to the console
    query = f"SELECT * FROM {input_table_name}"
    print(f"Executing query: {query}")
    table_env.execute_sql(query).print()
    
    if os.environ.get("IS_LOCAL"):
        # In a local environment, execute the job synchronously
        table_env.execute("Getting Started Job")
    else:
        # In a remote environment, execute the job asynchronously
        # and print the job status
        job_client = table_env.execute_async("Getting Started Job")
        print(job_client.get_job_status())


# Entry point for the script
if __name__ == "__main__":
    main()

Successfully create table
Executing query: SELECT * FROM input_table


Py4JJavaError: An error occurred while calling o180.executeSql.
: org.apache.flink.table.api.ValidationException: Unable to create a source for reading table 'default_catalog.default_database.input_table'.

Table options are:

'aws.region'='us-east-1'
'connector'='kinesis'
'format'='json'
'json.timestamp-format.standard'='ISO-8601'
'scan.stream.initpos'='TRIM_HORIZON'
'stream'='input-stream'
	at org.apache.flink.table.factories.FactoryUtil.createDynamicTableSource(FactoryUtil.java:234)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.createDynamicTableSource(CatalogSourceTable.java:176)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.toRel(CatalogSourceTable.java:116)
	at org.apache.calcite.sql2rel.SqlToRelConverter.toRel(SqlToRelConverter.java:4060)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertIdentifier(SqlToRelConverter.java:2928)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertFrom(SqlToRelConverter.java:2482)
	at org.apache.cal

In [29]:
!pwd

/Users/codebase/Documents/codebase/courses/learn-flink/src/learn_flink


I0000 00:00:1793652107.137530 3171531 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


In [19]:
import os

jar_path = os.path.abspath("flink-sql-connector-kinesis-1.16.1.jar")

print("JAR exists:", os.path.isfile(jar_path))
print("JAR path:", jar_path)

JAR exists: True
JAR path: /Users/codebase/Documents/codebase/courses/learn-flink/src/learn_flink/flink-sql-connector-kinesis-1.16.1.jar


None
